In [ ]:
import os

os.chdir('mcp-net') # move to the project's directory root

In [ ]:
!pip install nnunetv2 # install the nnunet library

In [ ]:
# patch the package to add determinism
!python acdc_baselines/patch_nnunet_determinism.py

Due to limited GPU resources <br>
we decided to train nnUnet using the 250 epoch configuration


In [ ]:

# copy the trainer to the variants folder
# Add the trainer with fixed seed
import nnunetv2.training.nnUNetTrainer
import os
import shutil

variants_dir = os.path.join(os.path.dirname(nnunetv2.training.nnUNetTrainer.__file__), "variants")
print(variants_dir)
print("exists:", os.path.exists(variants_dir))
shutil.copy("acdc_baselines/nnunet_trainer_seeded_250epoch.py", variants_dir)

In [ ]:
# verify that the trainer was added successfully
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
import nnunetv2.training.nnUNetTrainer, os
print(recursive_find_python_class(
    os.path.dirname(nnunetv2.training.nnUNetTrainer.__file__),
    "nnUNetTrainerSeeded_250epochs",
    "nnunetv2.training.nnUNetTrainer",
))

# output should be something like: <class 'nnunetv2.training.nnUNetTrainer.variants.nnunet_trainer_seeded_250epoch.nnUNetTrainerSeeded_250epochs'>


# Preparing ACDC Data for nnUnet

In [ ]:
# convert ACDC to nnUnet expected format
!python acdc_baselines/prepare_acdc_for_nnunet.py

In [ ]:
# preprocess using nnUnet preprocessor
import os

NNUNET_RAW = "data/nnUnet/nnUNet_raw"
NNUNET_PREPROCESSED = "data/nnUnet/nnUNet_preprocessed"
NNUNET_RESULTS = "data/nnUnet/nnUNet_results"

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

!nnUNetv2_plan_and_preprocess -d 27 --verify_dataset_integrity

# Training nnUnet

In [ ]:
# train with 250 epoch
import os

NNUNET_RAW = "data/nnUnet/nnUNet_raw"
NNUNET_PREPROCESSED = "data/nnUnet/nnUNet_preprocessed"
NNUNET_RESULTS = "data/nnUnet/nnUNet_results"

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS
!nnUNetv2_train 27 2d 0 -tr nnUNetTrainerSeeded_250epochs --c

# Predict on ACDC

In [ ]:
# predict on test set
!nnUNetv2_predict \
  -i '{NNUNET_RAW}/Dataset027_ACDC/mcpnet_test_heldout_images' \
  -o 'data/nnUnet/nnunet_predictions' \
  -d 27 \
  -c 2d \
  -f 0 \
  -tr nnUNetTrainerSeeded_250epochs

# Prepare M&Ms for nnUnet

In [ ]:
!python scripts/prepare_mms_for_nnunet.py

# nnUnet inference on M&Ms

In [ ]:
!nnUNetv2_predict \
-i 'data/M&Ms/mms_for_nnunet/images' \
-o  'data/nnUney/M&Ms/nnunet_mms_predictions'\
-d 27 \
-c 2d \
-f 0 \
-tr nnUNetTrainerSeeded_250epochs

# Benchmark Performance

In [ ]:
!pip install thop==0.1.1

In [ ]:
import time

import numpy as np
import torch
from torch.utils.flop_counter import FlopCounterMode



def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_flops(model, input_shape, device):
    
    dummy = torch.randn(1, *input_shape).to(device)
    model.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fcm:
        model(dummy)
    return fcm.get_total_flops()


def benchmark_inference_time(model, input_shape, device, n_warmup=10, n_runs=50):
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(dummy)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - start)

    return np.mean(times), np.std(times)


def benchmark_peak_memory(model, input_shape, device):
    if device.type != "cuda":
        return None  # peak memory tracking only meaningful on GPU here
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)
    torch.cuda.reset_peak_memory_stats(device)
    with torch.no_grad():
        _ = model(dummy)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB


def benchmark_pytorch_model(model, input_shape, model_name, device=None, slices_per_volume=10):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    n_params = count_params(model)
    flops = count_flops(model, input_shape, device)
    mean_time_per_slice, std_time_per_slice = benchmark_inference_time(model, input_shape, device)
    peak_mem_mb = benchmark_peak_memory(model, input_shape, device)

    print(f"=== {model_name} ===")
    print(f"Parameters: {n_params:,}")
    print(f"FLOPs: {flops/1e9:.3f} GFLOPs")
    print(f"Inference time per slice: {mean_time_per_slice*1000:.2f} ± {std_time_per_slice*1000:.2f} ms")
    print(f"Extrapolated time per volume ({slices_per_volume} slices, sequential): "
          f"{mean_time_per_slice*slices_per_volume*1000:.2f} ms")
    if peak_mem_mb is not None:
        print(f"Peak GPU memory: {peak_mem_mb:.1f} MB")
    else:
        print("Peak GPU memory: N/A (running on CPU)")
    print(f"Device: {device}")
    print()

    return {
        "model_name": model_name, "params": n_params, "flops": flops,
        "mean_time_per_slice_ms": mean_time_per_slice * 1000,
        "std_time_per_slice_ms": std_time_per_slice * 1000,
        "time_per_volume_ms": mean_time_per_slice * slices_per_volume * 1000,
        "peak_memory_mb": peak_mem_mb, "device": str(device),
    }

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import torch

predictor = nnUNetPredictor(
    tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
predictor.initialize_from_trained_model_folder(
    model_training_output_dir="/content/drive/MyDrive/Colab_Notebooks/mcp-net/other-models-data/nnUNet_results/Dataset027_ACDC/nnUNetTrainerSeeded_250epochs__nnUNetPlans__2d",
    use_folds=(0,),
    checkpoint_name="checkpoint_final.pth",
)

nnunet_model = predictor.network  # the actual nn.Module — this is what you pass to the benchmark
benchmark_pytorch_model(nnunet_model, (1,128,128), "nnU-Net")